# Unit 3: RNN 理论基础

## 学习目标
- 理解序列数据的特点和挑战
- 掌握 RNN Cell 的数学原理
- 理解 BPTT (Backpropagation Through Time)
- 认识梯度消失和梯度爆炸问题
- 用纯 NumPy/PyTorch 手动实现一个 RNN

## 3.1 为什么需要 RNN？

传统全连接网络处理的是固定大小的输入（如图像 224x224），但很多数据是**序列**：

- 文本："我 爱 深度 学习"
- 语音：连续的音频采样点
- 股票价格：每日收盘价时间序列
- 视频：连续的图像帧

序列数据的核心挑战：**长度可变** 且 **时间维度上的依赖关系**。

RNN 的核心思想：**参数共享** — 同一个 Cell 在序列的每个时间步重复使用，通过隐藏状态（hidden state）传递时序信息。

## 3.2 RNN Cell 数学公式

经典 RNN (Elman Network) 在每个时间步 $t$ 执行以下计算：

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

其中：
- $x_t \in \mathbb{R}^{d_{in}}$: 时间步 $t$ 的输入
- $h_t \in \mathbb{R}^{d_h}$: 时间步 $t$ 的隐藏状态（RNN 的"记忆"）
- $h_{t-1}$: 上一时间步的隐藏状态
- $W_{xh}$: 输入 → 隐藏 的权重矩阵
- $W_{hh}$: 隐藏 → 隐藏 的权重矩阵 (递归连接)
- $W_{hy}$: 隐藏 → 输出 的权重矩阵

**关键点：** $W_{xh}$、$W_{hh}$、$W_{hy}$ 在所有时间步**共享**，这就是 RNN 能处理变长序列的原因。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 3.3 手动实现 RNN Cell

我们先用纯 PyTorch 操作实现一个 RNN Cell，加深理解。  
- torch.rand(...) * 2 * k - k 将 [0, 1) 的随机数映射到 [-k, k) 区间。这种初始化方式能保证前向传播时信号方差稳定，避免梯度消失或爆炸。
- torch.tanh(...): 使用双曲正切激活函数将结果压缩到 (-1, 1) 之间，引入非线性并防止数值无限增长。

In [ ]:
class SimpleRNNCell:
    """手动实现 RNN Cell: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)"""
    def __init__(self, input_size, hidden_size):
        # 初始化权重 (使用 Xavier 初始化)
        k = 1.0 / (hidden_size ** 0.5)
        self.W_xh = torch.rand(input_size, hidden_size) * 2 * k - k
        self.W_hh = torch.rand(hidden_size, hidden_size) * 2 * k - k
        self.b_h = torch.zeros(hidden_size)

        # 需要梯度
        for p in [self.W_xh, self.W_hh, self.b_h]:
            p.requires_grad = True

    def forward(self, x_t, h_prev):
        # h_t = tanh(x_t @ W_xh + h_{t-1} @ W_hh + b_h)
        h_t = torch.tanh(x_t @ self.W_xh + h_prev @ self.W_hh + self.b_h)
        return h_t

    def parameters(self):
        return [self.W_xh, self.W_hh, self.b_h]

# 测试
cell = SimpleRNNCell(input_size=4, hidden_size=3)
x_t = torch.randn(4)      # 当前输入
h_prev = torch.zeros(3)   # 初始隐藏状态
h_t = cell.forward(x_t, h_prev)
print(f"x_t shape: {x_t.shape}")
print(f"h_prev shape: {h_prev.shape}")
print(f"h_t shape: {h_t.shape}")
print(f"h_prev values: {h_prev}")
print(f"h_t values: {h_t}")

## 3.4 完整的 RNN 前向传播

将 RNN Cell 应用于整个序列，计算所有时间步的输出：  

| 参数 | Shape | 说明 |
| :--- | :--- | :--- |
| x_seq | (seq_len, input_size) | 完整输入序列，`注意这里没有 batch 维度` |
| h0 | (hidden_size,) | 初始隐藏状态（一维，单样本） |
| W_xh | (input_size, hidden_size) | 输入→隐藏 权重 |
| W_hh | (hidden_size, hidden_size) | 隐藏→隐藏 循环权重 |
| b_h | (hidden_size,) | 隐藏层偏置 |
| W_hy | (hidden_size, output_size) | 新增：隐藏→输出 权重 |
| b_y | (output_size,) | 新增：输出层偏置 |
  
5 个时间步、`3 维输入`、4 个隐藏单元、2 维输出


In [ ]:
def rnn_forward(x_seq, h0, W_xh, W_hh, b_h, W_hy, b_y):
    """
    完整的 RNN 前向传播
    Args:
        x_seq: [seq_len, input_size] 输入序列
        h0:    [hidden_size] 初始隐藏状态
    Returns:
        outputs: [seq_len, output_size] 每个时间步的输出序列
        h_states: [seq_len, hidden_size] 每个时间步的隐藏状态
    """
    seq_len = x_seq.shape[0]
    hidden_size = h0.shape[0]
    output_size = b_y.shape[0]

    h_states = torch.zeros(seq_len, hidden_size)
    outputs = torch.zeros(seq_len, output_size)

    # h_t 在循环中被反复覆盖：每次迭代后 h_t 变为新值
    # 下一次迭代时作为 h_prev 参与计算，这正是 RNN "循环"特性的代码体现。
    h_t = h0

    # 这段代码的时间步遍历方式
    for t in range(seq_len):
        h_t = torch.tanh(x_seq[t] @ W_xh + h_t @ W_hh + b_h)
        y_t = h_t @ W_hy + b_y
        h_states[t] = h_t
        outputs[t] = y_t

    return outputs, h_states

# 参数设置
seq_len, input_size, hidden_size, output_size = 5, 3, 4, 2

# 随机初始化和输入
torch.manual_seed(42)
x_seq = torch.randn(seq_len, input_size)
h0 = torch.zeros(hidden_size)

W_xh = torch.randn(input_size, hidden_size) * 0.1
W_hh = torch.randn(hidden_size, hidden_size) * 0.1
b_h = torch.zeros(hidden_size)
W_hy = torch.randn(hidden_size, output_size) * 0.1
b_y = torch.zeros(output_size)

outputs, h_states = rnn_forward(x_seq, h0, W_xh, W_hh, b_h, W_hy, b_y)
print(f"输入序列 shape: {x_seq.shape}")
print(f"输出序列 shape: {outputs.shape}")
print(f"隐藏状态 shape: {h_states.shape}")
print(f"\n输入序列:\n{x_seq}")
print(f"\n隐藏状态 (随时间演变):\n{h_states}")
print(f"\n输出:\n{outputs}")

## 3.5 可视化 RNN 的隐藏状态演变

通过一个简单的序列预测任务来观察隐藏状态如何随时间变化。

| 索引 `i` | 输入 `X[i]` | 标签 `y[i]` |
| :--- | :--- | :--- |
| 0 | `data[0:20]` | `data[20]` |
| 1 | `data[1:21]` | `data[21]` |
| 2 | `data[2:22]` | `data[22]` |
| ... | ... | ... |
| 179 | `data[179:199]` | `data[199]` |



In [ ]:
# 创建一个简单的正弦波序列预测任务
t = np.linspace(0, 4 * np.pi, 200)
sin_wave = np.sin(t)

plt.figure(figsize=(12, 3))
plt.plot(t, sin_wave, linewidth=1)
# 正弦波序列 (前150点为训练，后50点为测试)
plt.title("Sine Wave Sequence (First 150 points for training, last 50 points for testing)")
plt.xlabel("Time step")
plt.ylabel("Value")
plt.axvline(x=t[150], color='r', linestyle='--', label="Training/Test Split")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


这段代码定义了一个用于正弦波预测的标准 RNN 回归模型。它展示了 PyTorch 中序列建模最核心的范式：RNN编码器 + 全连接输出头。
```text
输入 x: (batch, seq_len=20, input_size=1)
        │
        ▼
   ┌─────────┐
   │ nn.RNN  │  → out: (batch, seq_len, hidden_size=32)
   │         │  → h_n: (num_layers, batch, hidden_size=32)
   └────┬────┘
        │
   out[:, -1, :]  ← 只取最后一个时间步 (batch, 32)
        │
        ▼
   ┌─────────┐
   │ nn.Linear│  → (batch, 1)
   └────┬────┘
        │
    .squeeze(-1)  → (batch,)  ← 最终预测值
```

| 组件 | 参数 | 作用 |
| :--- | :--- | :--- |
| `nn.RNN` | `input_size=1` | 每个时间步输入1个标量（正弦波值） |
| | `hidden_size=32` | 隐状态维度，决定模型的记忆容量 |
| | `num_layers=1` | 单层RNN，对简单正弦波已足够 |
| | `batch_first=True` | ⚠️ 关键参数：指定输入格式为 `(batch, seq, feature)` 而非默认的 `(seq, batch, feature)` |
| `nn.Linear` | `32 → 1` | 将32维隐状态映射回1维标量预测值 |



💡 为什么用 batch_first=True？  
在数据处理阶段我们构建的张量形状是 (samples, seq_len, 1)，样本维度在前。设置此参数后，RNN 直接接受这种格式，无需手动 permute/transpose，大幅减少维度错误风险。PyTorch 默认 batch_first=False 是历史遗留设计（源于早期 Fortran/Lua 惯例），实际工程中几乎总是设为 True。


| 张量维度 | 形状符号 | 对应参数 | 本例中的值 | 含义 |
| :--- | :--- | :--- | :--- | :--- |
| Dim 0 | `batch` | - | 130 / 50 | 一次并行处理的样本数 |
| Dim 1 | `seq` | `seq_len` | 20 | 每个样本的时间步长度 |
| Dim 2 | `feature` | `input_size` | 1 | 每个时间步的特征向量维度 |



| 返回值 | 形状 (`batch_first=True`) | 包含的时间步 | 本质含义 |
| :--- | :--- | :--- | :--- |
| `out` | `(batch, seq_len, hidden_size)` | 所有时间步 | 最后一层在每个时间步的输出序列 |
| `h_n` | `(num_layers, batch, hidden_size)` | 仅最后一个时间步 | 每一层在序列结束时的最终隐状态 |



```text
时间步:    t=0      t=1      t=2     ...    t=19
          ┌───┐    ┌───┐    ┌───┐          ┌───┐
Layer 1:  │h₁₀│ → │h₁₁│ → │h₁₂│ → ... → │h₁₁₉│ ← h_n[0]
          └─┬─┘    └─┬─┘    └─┬─┘          └─┬─┘
            ↓        ↓        ↓                ↓
          ┌───┐    ┌───┐    ┌───┐          ┌───┐
Layer 2:  │h₂₀│ → │h₂₁│ → │h₂₂│ → ... → │h₂₁₉│ ← h_n[1]
          └─┬─┘    └─┬─┘    └─┬─┘          └─┬─┘
            ↓        ↓        ↓                ↓
           out      out      out              out
         [:,0,:]  [:,1,:]  [:,2,:]        [:,19,:]
         
←──────────── out 包含所有这些 ────────────────→
                                              ↑
                                         h_n 只取这里
```

- out 只包含最后一层的输出
- h_n 包含所有层的最终状态

In [ ]:
# 用 nn.RNN 快速训练一个正弦波预测模型
class SineRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, h_n = self.rnn(x)  # x: (batch, seq_len, input_size)
        # out: (batch, seq_len, hidden_size)
        # h_n: (num_layers, batch, hidden_size)
        return self.fc(out[:, -1, :]), h_n  # 取最后时间步
        # return: (batch, 1), (num_layers, batch, hidden_size)

model = SineRNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


# 用 PyTorch RNN 拟合正弦波
# 创建输入序列: 用前 seq_len 个点预测下一个点
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        # 切片操作始终保留原始维度。即使你只切了1个元素
        X.append(data[i:i+seq_len])       # (seq_len, 1)
        # 整数索引会消除（squeeze）被索引的那个维度。这是所有张量库的通用设计。
        y.append(data[i+seq_len])          # (1,)  ← 自动保留特征维度
    return torch.stack(X), torch.stack(y)  # 用 stack 替代 np.array 转换

seq_len = 20
data = torch.tensor(sin_wave, dtype=torch.float32).unsqueeze(-1)  # [200, 1]
# X_all: (180, 20, 1)
# y_all: (180, 1)
X_all, y_all = create_sequences(data, seq_len)
print(f"X_all: {X_all.shape}, y_all: {y_all.shape}")

# 划分训练集和测试集
X_train, X_test = X_all[:130], X_all[130:]
y_train, y_test = y_all[:130], y_all[130:]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")


train_loss = []
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    pred, _ = model(X_train.to(device))
    loss = criterion(pred, y_train.to(device))
    loss.backward()
    optimizer.step()
    train_loss.append(loss.item())

print(f"Final Train Loss: {train_loss[-1]:.6f}")

# 测试
model.eval()
with torch.no_grad():
    test_pred, hidden_states = model(X_test.to(device))
    test_loss = criterion(test_pred, y_test.to(device))
    print(f"Test Loss: {test_loss.item():.6f}")

# 可视化预测
plt.figure(figsize=(12, 4))
plt.plot(range(len(y_train)), y_train, label="Train True", alpha=0.6)
test_start = len(y_train)
plt.plot(range(test_start, test_start+len(y_test)), y_test, label="Test True",  linewidth=2, alpha=1)
plt.plot(range(test_start, test_start+len(test_pred)), test_pred.cpu(), label="Predicted", linewidth=1, alpha=0.5)
plt.legend()
plt.title("RNN sin(x) Prediction")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.grid(True, alpha=0.3)
plt.show()

## 3.6 BPTT: 通过时间的反向传播

BPTT 是标准反向传播在时间维度上的展开：

1. **展开 (Unroll)** RNN 沿时间轴，得到一个深度为序列长度的前馈网络
2. **反向传播** 梯度从最后一个时间步一路传到第一个时间步
3. 由于参数共享，每个时间步的梯度会**求和**

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial W_{hh}}
$$

**截断 BPTT (Truncated BPTT)**：对于超长序列，每 K 步截断一次，减少计算量。

⚠️ 一个容易被忽略的细节  
在这个特定的 compute_gradient_norm 函数中，h 是用 torch.zeros() 创建的，没有设置 requires_grad=True。
这意味着初始隐状态被视为一个常数而非可学习参数。在反向传播计算 x.grad 时，梯度只会沿着 x -> h_0 -> h_1 -> ... -> loss 这条路径回传，而不会尝试去优化初始状态本身。这完全符合该函数"研究输入序列梯度随长度变化规律"的设计意图。如果在实际训练中希望让网络自己学习最优的初始记忆，则需要将初始隐状态设为 nn.Parameter。  

为什么用范数而不是单个梯度值？
x.grad 是一个长度为 seq_len 的向量，不同时间步的梯度可能有正有负、相互抵消。L2 范数将所有时间步的梯度能量聚合为一个非负标量，消除了符号干扰，使得不同序列长度下的梯度强度具有可比性。这正是绘制"梯度范数 vs 序列长度"曲线所必需的。


In [ ]:
# 演示 BPTT 中的梯度流动
# 构建一个简单 RNN 序列，观察梯度大小变化

def compute_gradient_norm(seq_len, weight_scale=1.0):
    """计算不同序列长度下的梯度范数"""
    hidden_size = 10
    x = torch.randn(seq_len, 1, requires_grad=True)
    h = torch.zeros(1, hidden_size) # batch_size=1

    W_xh = torch.randn(1, hidden_size)
    W_hh = torch.randn(hidden_size, hidden_size) * weight_scale

    hs = []
    for t in range(seq_len):
        h = torch.tanh(x[t:t+1] @ W_xh + h @ W_hh)
        hs.append(h)

    loss = hs[-1].sum()
    loss.backward()
    # .norm()：计算梯度的 L2 范数
    return x.grad.norm().item()

# 对比不同 W_hh 的谱半径对梯度的影响
seq_lengths = [5, 10, 20, 30, 50]

for scale, label in [(0.5, "W_hh Spectrum Radius < 1 (Gradient Vanishing)"), (1.5, "W_hh Spectrum Radius > 1 (Gradient Explosion)")]:
    grads = []
    for L in seq_lengths:
        g = compute_gradient_norm(L, weight_scale=scale / np.sqrt(10))
        grads.append(g)

    plt.plot(seq_lengths, grads, 'o-', label=label, linewidth=2)

plt.xlabel("Sequence Length")
plt.ylabel("Gradient Norm at x[0]")
plt.title("BPTT Gradient Norm at x[0] vs Sequence Length")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3.7 梯度消失与梯度爆炸

### 梯度消失
- **原因**：$W_{hh}$ 的特征值 < 1，梯度在 BPTT 中指数衰减
- **后果**：无法学习长距离依赖
- **缓解**：LSTM/GRU（门控机制）、ReLU 激活、梯度裁剪

### 梯度爆炸
- **原因**：$W_{hh}$ 的特征值 > 1，梯度指数增长
- **后果**：训练不稳定，参数更新过大
- **缓解**：梯度裁剪（Gradient Clipping）— 将梯度范数限制在阈值内

In [ ]:
# 梯度裁剪演示
model_demo = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 1)
)

x = torch.randn(5, 10)
y = torch.randn(5, 1)

# 故意产生大梯度
pred = model_demo(x)
loss = nn.MSELoss()(pred, y)
loss.backward()

# 裁剪前
total_norm_before = sum(p.grad.norm().item() for p in model_demo.parameters() if p.grad is not None)
print(f"梯度裁剪前总范数: {total_norm_before:.4f}")

# 梯度裁剪: 将全局梯度范数限制在 max_norm
torch.nn.utils.clip_grad_norm_(model_demo.parameters(), max_norm=1.0)

total_norm_after = sum(p.grad.norm().item() for p in model_demo.parameters() if p.grad is not None)
print(f"梯度裁剪后总范数: {total_norm_after:.4f}")

## 要点总结

| 概念 | 核心思想 |
|------|----------|
| RNN Cell | $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$ |
| 参数共享 | 同一套权重在所有时间步复用 |
| 隐藏状态 | RNN 的"记忆"，随时间步传递 |
| BPTT | 沿时间轴展开的反向传播 |
| 梯度消失 | 长序列中早期梯度 → 0，难以学习长期依赖 |
| 梯度爆炸 | 梯度指数增长导致训练不稳定 |
| 梯度裁剪 | `clip_grad_norm_` 限制梯度范数 |

### 练习
1. 修改手动实现的 RNN，尝试用 ReLU 替代 Tanh，观察效果
2. 改变 `W_hh` 的初始化方式（如正交初始化），对比 BPTT 梯度
3. 用 `detach()` 实现截断 BPTT（每 10 步截断一次）